[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/02_joint_and_conditional_entropy/exercises.ipynb)

# Module 02 — Exercises: Joint and Conditional Entropy

Twenty-three solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it.

Theorem, lemma and proof numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): $H(X, Y)$ is joint entropy, logarithms are base
$2$ and every numeric answer carries the unit **bits**.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps


def H(p):
    """Shannon entropy in bits, with the 0 log 0 = 0 convention."""
    p = np.asarray(p, dtype=float).ravel()
    p = p[p > 0.0]
    return float(-(p * np.log2(p)).sum()) + 0.0


def Hb(t):
    """Binary entropy function in bits."""
    return H([t, 1.0 - t])


def entropies(P):
    """Joint table -> (H(X,Y), H(X), H(Y), H(Y|X), H(X|Y)) in bits."""
    P = np.asarray(P, dtype=float)
    Hxy, Hx, Hy = H(P), H(P.sum(axis=1)), H(P.sum(axis=0))
    return Hxy, Hx, Hy, Hxy - Hx, Hxy - Hy


KB, TEMP = 1.380649e-23, 300.0          # J/K and K, used by the physics problems
print(f"machine epsilon = {EPS:.4e}")
print(f"kB T ln 2 at {TEMP:.0f} K = {KB * TEMP * np.log(2.0):.4e} J per erased bit")

machine epsilon = 2.2204e-16
kB T ln 2 at 300 K = 2.8710e-21 J per erased bit


## L0 — Concept Checks

### Problem L0.1 — Joint entropy of two independent coins

**Statement.** Two independent fair coins $X$ and $Y$ are flipped. Give $H(X)$, $H(Y)$ and
$H(X, Y)$ in bits.

**Intuition.** Independent flips carry separate information, so the costs add.

**Solution.**

*Step 1.* Each coin is uniform on two outcomes, so $H(X) = H(Y) = \log 2 = 1$ bit.

*Step 2.* By independence the pair is uniform on four outcomes, so
$H(X, Y) = \log 4 = 2$ bits.

$$
\boxed{H(X) = H(Y) = 1 \text{ bit}, \qquad H(X,Y) = 2 \text{ bits}}
$$

**Key takeaway.** This is the equality case of Theorem 4.5: entropy is exactly additive over
independent variables and strictly subadditive otherwise.

In [2]:
P = np.full((2, 2), 0.25)
Hxy, Hx, Hy, Hy_x, _ = entropies(P)
print(f"H(X) = {Hx:.4f}  H(Y) = {Hy:.4f}  H(X,Y) = {Hxy:.4f}  H(Y|X) = {Hy_x:.4f}")
assert (abs(Hx - 1) < 1e-12) and (abs(Hy - 1) < 1e-12) and (abs(Hxy - 2) < 1e-12)

H(X) = 1.0000  H(Y) = 1.0000  H(X,Y) = 2.0000  H(Y|X) = 1.0000


### Problem L0.2 — Joint entropy of a perfect copy

**Statement.** Let $X$ be a fair coin and $Y = X$. Give $H(X, Y)$ and $H(Y \mid X)$.

**Intuition.** A copy carries no new information, so the pair costs what one coin costs.

**Solution.**

*Step 1.* The pair takes only the two values $(0,0)$ and $(1,1)$, each with probability
$\tfrac12$, so $H(X, Y) = \log 2 = 1$ bit.

*Step 2.* $Y$ is a function of $X$, so $H(Y \mid X) = 0$ by Theorem 4.7.

*Step 3.* Chain-rule check: $H(X) + H(Y \mid X) = 1 + 0 = 1 = H(X,Y)$.

$$
\boxed{H(X, Y) = 1 \text{ bit}, \qquad H(Y \mid X) = 0}
$$

**Key takeaway.** The lower end of Proposition 4.6: $H(X,Y) = \max\{H(X), H(Y)\}$ exactly when one
variable determines the other.

In [3]:
P = np.array([[0.5, 0.0], [0.0, 0.5]])
Hxy, Hx, Hy, Hy_x, _ = entropies(P)
print(f"H(X,Y) = {Hxy:.4f}  H(X) = {Hx:.4f}  H(Y|X) = {Hy_x:.4f}")
assert abs(Hxy - 1) < 1e-12 and abs(Hy_x) < 1e-12 and abs(Hxy - max(Hx, Hy)) < 1e-12

H(X,Y) = 1.0000  H(X) = 1.0000  H(Y|X) = 0.0000


### Problem L0.3 — Ordering the three quantities

**Statement.** Without computing anything, order $H(Y)$, $H(Y \mid X)$ and $H(X, Y)$ for an
arbitrary pair, and say when each inequality is tight.

**Intuition.** Conditioning can only shrink; adjoining a variable can only grow.

**Solution.**

*Step 1.* $H(Y \mid X) \le H(Y)$ by Theorem 4.4(a), tight exactly at independence.

*Step 2.* $H(X, Y) = H(Y) + H(X \mid Y) \ge H(Y)$ by Theorem 4.1 and non-negativity of
conditional entropy, tight exactly when $H(X \mid Y) = 0$, i.e. when $X = f(Y)$ (Theorem 4.7).

$$
\boxed{H(Y \mid X) \le H(Y) \le H(X, Y)}
$$

**Key takeaway.** Residual uncertainty is at most marginal uncertainty, which is at most joint
uncertainty, with independence and determinism as the two boundary regimes.

### Problem L0.4 — Conditional entropy is directional

**Statement.** Let $X$ be uniform on $\{1,2,3,4\}$ and $Y = \mathbf{1}\{X \text{ is even}\}$.
Compute $H(Y \mid X)$ and $H(X \mid Y)$.

**Intuition.** Parity is computable from the number; the number is not recoverable from its
parity.

**Solution.**

*Step 1.* $Y = f(X)$, so $H(Y \mid X) = 0$ by Theorem 4.7.

*Step 2.* $Y$ is a fair bit, and each value of $Y$ leaves $X$ uniform on two candidates, so
$H(X \mid Y) = \log 2 = 1$ bit.

*Step 3.* Chain-rule check: $H(X,Y) = H(X) = 2$ bits since the pair is determined by $X$; and
$H(Y) + H(X \mid Y) = 1 + 1 = 2$.

$$
\boxed{H(Y \mid X) = 0 \ \neq \ H(X \mid Y) = 1 \text{ bit}}
$$

**Key takeaway.** Conditional entropy is not symmetric; only the *difference*
$H(X) - H(X \mid Y) = H(Y) - H(Y \mid X)$ is (Proof 5.1).

In [4]:
P = np.zeros((4, 2))
for i, v in enumerate([1, 2, 3, 4]):
    P[i, int(v % 2 == 0)] = 0.25      # column 1 is the event "v is even"
Hxy, Hx, Hy, Hy_x, Hx_y = entropies(P)
print(f"H(X) = {Hx:.4f}  H(Y) = {Hy:.4f}  H(X,Y) = {Hxy:.4f}")
print(f"H(Y|X) = {Hy_x:.4f}   H(X|Y) = {Hx_y:.4f}")
assert abs(Hy_x) < 1e-12 and abs(Hx_y - 1.0) < 1e-12 and abs(Hxy - 2.0) < 1e-12

H(X) = 2.0000  H(Y) = 1.0000  H(X,Y) = 2.0000
H(Y|X) = 0.0000   H(X|Y) = 1.0000


### Problem L0.5 — Chain rule for three fair bits

**Statement.** Expand $H(X_1, X_2, X_3)$ by the chain rule in the order $1, 2, 3$, and evaluate it
for three i.i.d. fair bits.

**Intuition.** Reveal the bits one at a time; independence makes each conditional term a marginal.

**Solution.**

*Step 1.* Theorem 4.2 gives
$H(X_1, X_2, X_3) = H(X_1) + H(X_2 \mid X_1) + H(X_3 \mid X_1, X_2)$.

*Step 2.* For i.i.d. fair bits every conditional term equals its marginal, $1$ bit each, so the
sum is $3$ bits — matching the direct count $\log 8 = 3$.

$$
\boxed{H(X_1,X_2,X_3) = H(X_1) + H(X_2 \mid X_1) + H(X_3 \mid X_1,X_2) = 3 \text{ bits}}
$$

**Key takeaway.** The chain rule holds for every ordering; independence is what collapses each
conditional term to a marginal.

In [5]:
J = np.full((2, 2, 2), 0.125)
H123 = H(J)
H1 = H(J.sum((1, 2)))
H12 = H(J.sum(2))
print(f"H(X1) = {H1:.4f}  H(X2|X1) = {H12 - H1:.4f}  H(X3|X1,X2) = {H123 - H12:.4f}")
print(f"sum = {H1 + (H12 - H1) + (H123 - H12):.4f}   direct H = {H123:.4f}")
assert abs(H123 - 3.0) < 1e-12

H(X1) = 1.0000  H(X2|X1) = 1.0000  H(X3|X1,X2) = 1.0000
sum = 3.0000   direct H = 3.0000


## L1 — Foundations

### Problem L1.1 — Full computation from a joint table

**Statement.** Let $(X, Y)$ have $p(0,0) = \tfrac12$, $p(0,1) = \tfrac14$, $p(1,0) = 0$,
$p(1,1) = \tfrac14$. Compute $H(X, Y)$, $H(X)$, $H(Y)$, $H(Y \mid X)$ and $H(X \mid Y)$ in bits.

**Intuition.** Get the joint and both marginals; the chain rule then hands you both conditionals
without ever forming a conditional distribution.

**Solution.**

*Step 1 — joint.* Three non-zero cells with masses $\tfrac12, \tfrac14, \tfrac14$:

$$
H(X, Y) = \tfrac12 \log 2 + \tfrac14 \log 4 + \tfrac14 \log 4 = 1.5 \text{ bits}.
$$

*Step 2 — marginals.* $p_X = (\tfrac34, \tfrac14)$ and $p_Y = (\tfrac12, \tfrac12)$, so

$$
H(X) = H_b(\tfrac14) = 2 - \tfrac34 \log 3 = 0.811278, \qquad H(Y) = 1 .
$$

*Step 3 — conditionals by Theorem 4.1.*

$$
H(Y \mid X) = 1.5 - 0.811278 = 0.688722, \qquad H(X \mid Y) = 1.5 - 1 = 0.5 .
$$

*Step 4 — sanity.* Both conditionals sit below their marginals, as Theorem 4.4 requires.

$$
\boxed{H(X,Y) = 1.5, \quad H(X) = 0.811278, \quad H(Y) = 1, \quad H(Y \mid X) = 0.688722, \quad H(X \mid Y) = 0.5 \ \text{(bits)}}
$$

**Key takeaway.** One joint-entropy computation plus two marginals yields every conditional
entropy for free — and is numerically safer than dividing by small marginals.

In [6]:
P = np.array([[0.5, 0.25], [0.0, 0.25]])
Hxy, Hx, Hy, Hy_x, Hx_y = entropies(P)
px = P.sum(axis=1)
slice_avg = sum(px[i] * H(P[i] / px[i]) for i in range(2) if px[i] > 0)
print(f"H(X,Y) = {Hxy:.6f}   H(X) = {Hx:.6f}   H(Y) = {Hy:.6f}")
print(f"H(Y|X) = {Hy_x:.6f} (chain rule) = {slice_avg:.6f} (slice average)")
print(f"H(X|Y) = {Hx_y:.6f}")
assert abs(Hxy - 1.5) < 1e-12 and abs(Hx - Hb(0.25)) < 1e-12
assert abs(Hy_x - 0.688722) < 1e-6 and abs(Hx_y - 0.5) < 1e-12
assert abs(slice_avg - Hy_x) < 1e-12

H(X,Y) = 1.500000   H(X) = 0.811278   H(Y) = 1.000000
H(Y|X) = 0.688722 (chain rule) = 0.688722 (slice average)
H(X|Y) = 0.500000


### Problem L1.2 — Entropy of a Markov weather chain

**Statement.** A weather chain has states sunny and rainy, switches with probability $0.1$ each
day, and starts from the uniform distribution. Compute $H(X_1, \dots, X_n)$ and the entropy rate.

**Intuition.** After the first day, each new day costs only the entropy of one coin flip with bias
$0.1$.

**Solution.**

*Step 1.* By Theorem 4.2 and the Markov property, $H(X_t \mid X_{\lt t}) = H(X_t \mid X_{t-1})$,
so

$$
H(X_1, \dots, X_n) = H(X_1) + \sum_{t=2}^{n} H(X_t \mid X_{t-1}) .
$$

*Step 2.* The start is uniform, hence stationary for this symmetric chain, so $H(X_1) = 1$ bit and
each transition slice is $\mathrm{Bernoulli}(0.1)$ regardless of the current state:

$$
H(X_t \mid X_{t-1}) = H_b(0.1) = 0.468996 \text{ bits}.
$$

*Step 3.* Therefore $H(X_{1:n}) = 1 + (n-1)(0.468996)$ bits, and by Theorem 4.10

$$
H_\infty = \lim_n \tfrac1n H(X_{1:n}) = H_b(0.1) = 0.468996 \text{ bits/day}.
$$

$$
\boxed{H(X_{1:n}) = 1 + 0.468996\,(n-1) \text{ bits}, \qquad H_\infty = 0.468996 \text{ bits/day}}
$$

**Key takeaway.** For a stationary Markov chain the entropy rate is one step of conditional
entropy; persistence makes trajectories far more compressible than $n$ bits.

In [7]:
Pm = np.array([[0.9, 0.1], [0.1, 0.9]])
pi = np.array([0.5, 0.5])
rate = float(pi @ np.array([H(Pm[0]), H(Pm[1])]))
print(f"Hb(0.1) = {Hb(0.1):.6f} bits    entropy rate = {rate:.6f} bits/day")
for n in (2, 5, 10, 100):
    # exact joint entropy of the trajectory, built by the chain rule
    print(f"  n = {n:>3}: H(X_1..n) = {1 + (n - 1) * rate:9.4f} bits"
          f"   vs {n} bits for i.i.d. fair days")
assert np.allclose(pi @ Pm, pi)
assert abs(rate - Hb(0.1)) < 1e-12

Hb(0.1) = 0.468996 bits    entropy rate = 0.468996 bits/day
  n =   2: H(X_1..n) =    1.4690 bits   vs 2 bits for i.i.d. fair days
  n =   5: H(X_1..n) =    2.8760 bits   vs 5 bits for i.i.d. fair days
  n =  10: H(X_1..n) =    5.2210 bits   vs 10 bits for i.i.d. fair days
  n = 100: H(X_1..n) =   47.4306 bits   vs 100 bits for i.i.d. fair days


### Problem L1.3 — A slice where conditioning increases uncertainty

**Statement.** Construct $(X, Y)$ and a value $x$ with $H(Y \mid X{=}x) \gt H(Y)$, confirming
that Theorem 4.4 constrains only the average.

**Intuition.** Make the informative slice common and the confusing slice rare.

**Solution.**

*Step 1 — the construction.* Let $\mathbb{P}(X = 1) = 0.1$. Given $X = 0$ set $Y = 0$; given
$X = 1$ let $Y$ be a fair bit.

*Step 2 — the marginal.* $\mathbb{P}(Y = 1) = 0.1 \times 0.5 = 0.05$, so
$H(Y) = H_b(0.05) = 0.286397$ bits.

*Step 3 — the slices.* $H(Y \mid X{=}0) = 0$ and $H(Y \mid X{=}1) = 1$ bit, and $1 \gt 0.286397$.

*Step 4 — the average obeys the theorem.*
$H(Y \mid X) = 0.9(0) + 0.1(1) = 0.1 \lt 0.286397$.

$$
\boxed{H(Y \mid X{=}1) = 1 \ \gt \ H(Y) = 0.286397 \ \gt \ H(Y \mid X) = 0.1 \ \text{(bits)}}
$$

**Key takeaway.** A surprising observation can genuinely raise your uncertainty; only the
expectation over observations is protected.

In [8]:
P = np.array([[0.9, 0.0], [0.05, 0.05]])
Hxy, Hx, Hy, Hy_x, _ = entropies(P)
slice1 = H(P[1] / P[1].sum())
print(f"H(Y) = {Hy:.6f}   H(Y|X=1) = {slice1:.6f}   H(Y|X) = {Hy_x:.6f}")
print(f"slice beats the marginal: {slice1 > Hy};  average does not: {Hy_x <= Hy}")
assert slice1 > Hy and Hy_x < Hy and abs(Hy_x - 0.1) < 1e-12

H(Y) = 0.286397   H(Y|X=1) = 1.000000   H(Y|X) = 0.100000
slice beats the marginal: True;  average does not: True


### Problem L1.4 — Two-way determinism is a bijection

**Statement.** Suppose $H(Y \mid X) = 0$ and $H(X \mid Y) = 0$ simultaneously. Determine the
relationship between $X$ and $Y$ and prove it.

**Intuition.** Each variable computes the other, so neither can merge outcomes the other keeps
apart.

**Solution.**

*Step 1.* By Theorem 4.7, $H(Y \mid X) = 0$ gives $Y = f(X)$ almost surely and
$H(X \mid Y) = 0$ gives $X = g(Y)$ almost surely.

*Step 2.* Composing, $X = g(f(X))$ almost surely. If $x_1, x_2$ are support points with
$f(x_1) = f(x_2)$, then $x_1 = g(f(x_1)) = g(f(x_2)) = x_2$, so $f$ is injective on the support of
$X$.

*Step 3.* Symmetrically $g$ is injective on the support of $Y$, and $g$ inverts $f$, so $f$ is a
bijection between the two supports.

*Step 4.* Entropy depends only on the multiset of probabilities, which a bijection preserves, so
$H(X) = H(Y)$; and by Theorem 4.1, $H(X, Y) = H(X) + 0 = H(X)$.

$$
\boxed{X \text{ and } Y \text{ are relabelings of each other: } H(X) = H(Y) = H(X, Y)}
$$

**Key takeaway.** Mutual functional dependence means identical information in different encodings
— and by Definition 3.5 the overlap is then the whole of both entropies.

In [9]:
p = rng.dirichlet(np.ones(6))


def joint_from_map(p, f, ncols):
    """Joint table of (X, f(X)) for X ~ p."""
    P = np.zeros((len(p), ncols))
    for x in range(len(p)):
        P[x, f(x)] += p[x]
    return P


bij = joint_from_map(p, lambda x: (5 * x + 2) % 6, 6)      # a permutation of {0..5}
merge = joint_from_map(p, lambda x: x % 2, 2)              # many-to-one: parity
for name, P in [("bijection f", bij), ("parity   f", merge)]:
    Hxy, Hx, Hy, Hy_x, Hx_y = entropies(P)
    print(f"{name}: H(X) = {Hx:.6f}  H(Y) = {Hy:.6f}  H(X,Y) = {Hxy:.6f}  "
          f"H(Y|X) = {Hy_x:.1e}  H(X|Y) = {Hx_y:.6f}")
Hxy, Hx, Hy, Hy_x, Hx_y = entropies(bij)
assert abs(Hy_x) < 1e-12 and abs(Hx_y) < 1e-12
assert abs(Hx - Hy) < 1e-12 and abs(Hx - Hxy) < 1e-12
_, Hx2, _, Hy_x2, Hx_y2 = entropies(merge)
assert abs(Hy_x2) < 1e-12 and Hx_y2 > 0.0

bijection f: H(X) = 1.914765  H(Y) = 1.914765  H(X,Y) = 1.914765  H(Y|X) = 0.0e+00  H(X|Y) = 0.000000
parity   f: H(X) = 1.914765  H(Y) = 0.904793  H(X,Y) = 1.914765  H(Y|X) = 0.0e+00  H(X|Y) = 1.009972


### Problem L1.5 — Entropy of a sum

**Statement.** Let $X, Y$ be independent fair bits and $Z = X + Y \in \{0,1,2\}$. Compute $H(Z)$
and $H(Z \mid X)$, and explain why $H(Z) \lt H(X, Y)$.

**Intuition.** Adding merges two of the four outcomes, and merging destroys information.

**Solution.**

*Step 1.* $Z$ has distribution $(\tfrac14, \tfrac12, \tfrac14)$, so

$$
H(Z) = \tfrac14 \log 4 + \tfrac12 \log 2 + \tfrac14 \log 4 = 1.5 \text{ bits}.
$$

*Step 2.* Given $X = x$, $Z = x + Y$ is a shifted fair bit, so every slice has entropy $1$ and
$H(Z \mid X) = 1$ bit.

*Step 3.* $Z = g(X, Y)$ is a function of the pair, so $H(Z \mid X, Y) = 0$ and by Theorem 4.1

$$
H(Z) \le H(Z, X, Y) = H(X, Y) + H(Z \mid X, Y) = H(X,Y) = 2 \text{ bits}.
$$

The inequality is strict because $g$ merges $(0,1)$ and $(1,0)$: $H(X, Y \mid Z) = 0.5 \gt 0$.

$$
\boxed{H(Z) = 1.5 \text{ bits}, \qquad H(Z \mid X) = 1 \text{ bit}}
$$

**Key takeaway.** A deterministic function can only lose entropy, and the loss is exactly
$H(X, Y \mid Z)$ — half a bit here, the bit that distinguished the two mixed outcomes, times its
probability $\tfrac12$.

In [10]:
J = np.zeros((2, 2, 3))
for x in (0, 1):
    for y in (0, 1):
        J[x, y, x + y] = 0.25
Hz = H(J.sum((0, 1)))
Hxy = H(J.sum(2))
Hxz = H(J.sum(1))
Hx = H(J.sum((1, 2)))
print(f"H(Z) = {Hz:.6f}   H(X,Y) = {Hxy:.6f}   H(Z|X) = {H(J.sum(1)) - Hx:.6f}")
print(f"H(X,Y|Z) = {H(J) - Hz:.6f}   loss H(X,Y) - H(Z) = {Hxy - Hz:.6f}")
assert abs(Hz - 1.5) < 1e-12 and abs(Hxz - Hx - 1.0) < 1e-12
assert abs((H(J) - Hz) - 0.5) < 1e-12

H(Z) = 1.500000   H(X,Y) = 2.000000   H(Z|X) = 1.000000
H(X,Y|Z) = 0.500000   loss H(X,Y) - H(Z) = 0.500000


### Problem L1.6 — Stationarity is necessary for the entropy rate

**Statement.** (a) Exhibit a process with $a_n = H(X_n \mid X_{\lt n})$ **not** non-increasing, and
say exactly which step of Proof 5.10 fails. (b) Exhibit a process of independent bits for which
$\frac1n H(X_{1:n})$ has no limit at all.

**Intuition.** Proof 5.10 shifts the conditioning window by one step; only stationarity licenses
that shift.

**Solution.**

*Step 1 — part (a).* Let $X_1$ be a fair bit and let $X_2$ be uniform on $\{1,2,3,4\}$,
independent of $X_1$. Then

$$
a_1 = H(X_1) = 1, \qquad a_2 = H(X_2 \mid X_1) = H(X_2) = 2 \gt a_1 .
$$

*Step 2 — where the proof fails.* Step (a) of Proof 5.10 writes
$H(X_{n+1} \mid X_2, \dots, X_n) = H(X_n \mid X_1, \dots, X_{n-1})$, an index shift that is
stationarity itself. Here $X_2$ and $X_1$ do not even have the same alphabet, so the shift is
unavailable and the monotonicity argument never starts.

*Step 3 — part (b).* Let the $X_t$ be independent with

$$
X_t \sim \mathrm{Bernoulli}(\tfrac12) \text{ if } t \in B, \qquad X_t \equiv 0 \text{ otherwise},
\qquad B = \bigcup_{k \ge 0} \{ t : 4^k \le t \lt 2\cdot 4^k \} .
$$

By independence $a_t = 1$ for $t \in B$ and $a_t = 0$ otherwise, so $(a_t)$ has no limit; and by
Theorem 4.2, $\frac1n H(X_{1:n}) = \lvert B \cap [1,n] \rvert / n$.

*Step 4 — the two subsequences.* Put $s_k = \sum_{j \le k} 4^j = (4^{k+1}-1)/3$, the count of
"on" indices below $4^{k+1}$. At the end of an on-block, $n = 2 \cdot 4^k - 1$, the ratio tends to
$\tfrac{(4/3)4^k}{2 \cdot 4^k} = \tfrac23$; at the end of the following off-block,
$n = 4^{k+1}-1$, it tends to $\tfrac{(4/3)4^k}{4 \cdot 4^{k}} = \tfrac13$.

$$
\boxed{a_1 = 1 \lt a_2 = 2; \qquad \liminf_n \tfrac1n H(X_{1:n}) = \tfrac13, \quad \limsup_n = \tfrac23}
$$

**Key takeaway.** Without stationarity there is no entropy rate to speak of — the "bits per
symbol" of a non-stationary source need not converge, so Theorem 4.10's hypothesis is not
bookkeeping.

In [11]:
print("(a) a_1 and a_2 for the two-step process")
print(f"    a_1 = {H([0.5, 0.5]):.4f}   a_2 = {H(np.full(4, 0.25)):.4f}   monotone: "
      f"{H(np.full(4, 0.25)) <= H([0.5, 0.5])}")
assert H(np.full(4, 0.25)) > H([0.5, 0.5])

print("\n(b) blocks of doubling length: (1/n) H(X_1..n) oscillates")
nmax = 4 ** 9
on = np.zeros(nmax + 1, dtype=bool)
k = 0
while 4 ** k <= nmax:
    lo, hi = 4 ** k, min(2 * 4 ** k, nmax + 1)
    on[lo:hi] = True
    k += 1
cum = np.cumsum(on)
print(f"    {'n':>9} {'(1/n) H':>9}")
for k in range(1, 9):
    for n, tag in [(2 * 4 ** k - 1, "end of on-block "), (4 ** (k + 1) - 1, "end of off-block")]:
        print(f"    {n:>9} {cum[n] / n:>9.4f}   {tag}")
ratios_hi = [cum[2 * 4 ** k - 1] / (2 * 4 ** k - 1) for k in range(4, 9)]
ratios_lo = [cum[4 ** (k + 1) - 1] / (4 ** (k + 1) - 1) for k in range(4, 9)]
print(f"\n    limsup -> {max(ratios_hi):.4f} (predicted 0.6667),"
      f"  liminf -> {min(ratios_lo):.4f} (predicted 0.3333)")
assert abs(max(ratios_hi) - 2 / 3) < 0.01 and abs(min(ratios_lo) - 1 / 3) < 0.01

(a) a_1 and a_2 for the two-step process
    a_1 = 1.0000   a_2 = 2.0000   monotone: False

(b) blocks of doubling length: (1/n) H(X_1..n) oscillates
            n   (1/n) H
            7    0.7143   end of on-block 
           15    0.3333   end of off-block
           31    0.6774   end of on-block 
           63    0.3333   end of off-block
          127    0.6693   end of on-block 
          255    0.3333   end of off-block
          511    0.6673   end of on-block 
         1023    0.3333   end of off-block
         2047    0.6668   end of on-block 
         4095    0.3333   end of off-block
         8191    0.6667   end of on-block 
        16383    0.3333   end of off-block
        32767    0.6667   end of on-block 
        65535    0.3333   end of off-block
       131071    0.6667   end of on-block 
       262143    0.3333   end of off-block

    limsup -> 0.6673 (predicted 0.6667),  liminf -> 0.3333 (predicted 0.3333)


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Per-token loss is conditional entropy

**Statement.** An autoregressive model whose conditionals are the *true* ones scores a sequence
with the token-level log loss. Show that its expected loss at position $t$ is
$H(X_t \mid X_{\lt t})$ and that its expected total loss is $H(X_1, \dots, X_n)$.

**Intuition.** The log loss is the surprise of the observed token; averaging surprise is exactly
what conditional entropy does.

**Solution.**

*Step 1.* The loss at position $t$ is $-\log p(X_t \mid X_{\lt t})$. Its expectation under the
joint law is

$$
\mathbb{E}\bigl[-\log p(X_t \mid X_{\lt t})\bigr] = -\sum_{x_{\le t}} p(x_{\le t}) \log p(x_t \mid x_{\lt t}) = H(X_t \mid X_{\lt t}),
$$

by Definition 3.3 applied with conditioning variable $X_{\lt t}$.

*Step 2.* Summing over $t$ and applying Theorem 4.2,

$$
\mathbb{E}\left[\sum_{t=1}^{n} -\log p(X_t \mid X_{\lt t})\right] = \sum_{t=1}^{n} H(X_t \mid X_{\lt t}) = H(X_1, \dots, X_n) .
$$

$$
\boxed{\mathbb{E}[\text{teacher-forcing loss}] = \sum_{t} H(X_t \mid X_{\lt t}) = H(X_{1:n})}
$$

**Key takeaway.** The floor of next-token-prediction loss is the joint entropy of the data — for a
stationary source, the entropy rate of Theorem 4.10. Everything above it is model error.

In [12]:
Pm = np.array([[0.8, 0.2], [0.4, 0.6]])
pi = np.array([0.4, 0.2]) / 0.6
rate = float(pi @ np.array([H(Pm[0]), H(Pm[1])]))
n = 200_000
seq = np.empty(n, dtype=int)
seq[0] = int(rng.random() > pi[0])
u = rng.random(n)
for t in range(1, n):
    seq[t] = int(u[t] > Pm[seq[t - 1], 0])
loss = float(np.mean(-np.log2(Pm[seq[:-1], seq[1:]])))
print(f"entropy rate H_inf         = {rate:.6f} bits/step")
print(f"measured teacher-forcing loss = {loss:.6f} bits/step   (n = {n})")
assert abs(loss - rate) < 5e-3

entropy rate H_inf         = 0.804936 bits/step
measured teacher-forcing loss = 0.801605 bits/step   (n = 200000)


### Problem L2.2 — Information gain with a three-way split

**Statement.** Twelve examples, six of class A and six of class B, are split by a categorical
feature into groups of sizes $4/4/4$ with class counts $(4A, 0B)$, $(0A, 4B)$ and $(2A, 2B)$.
Compute the information gain and the gain ratio, in bits.

**Intuition.** Two children are pure and one is a coin flip; only the coin flip still costs
anything.

**Solution.**

*Step 1 — parent.* Six of each class, so $H(Y) = H_b(\tfrac12) = 1$ bit.

*Step 2 — children.* Their entropies are $0$, $0$ and $1$ bit, each with weight $\tfrac13$:

$$
H(Y \mid X) = \tfrac13(0) + \tfrac13(0) + \tfrac13(1) = \tfrac13 \text{ bits}.
$$

*Step 3 — gain.* $\operatorname{IG} = 1 - \tfrac13 = \tfrac23 = 0.666667$ bits, non-negative as
Theorem 4.4 guarantees.

*Step 4 — gain ratio.* The split itself has entropy $H(X) = \log 3 = 1.584963$ bits, so

$$
\operatorname{GR} = \frac{2/3}{\log 3} = 0.420620 .
$$

$$
\boxed{\operatorname{IG} = \tfrac23 = 0.666667 \text{ bits}, \qquad \operatorname{GR} = 0.420620}
$$

**Key takeaway.** Information gain is the subadditivity gap; the gain ratio divides it by the
split entropy to stop many-valued features from winning by fragmentation alone.

In [13]:
children = np.array([[4.0, 0.0], [0.0, 4.0], [2.0, 2.0]])
tot = children.sum()
w = children.sum(axis=1) / tot
Hy = H(children.sum(axis=0) / tot)
Hy_x = float(sum(w[i] * H(children[i] / children[i].sum()) for i in range(3)))
Hx = H(w)
print(f"H(Y) = {Hy:.6f}   H(Y|X) = {Hy_x:.6f}   IG = {Hy - Hy_x:.6f}")
print(f"H(X) = {Hx:.6f}   gain ratio = {(Hy - Hy_x) / Hx:.6f}")
assert abs(Hy - 1.0) < 1e-12 and abs(Hy_x - 1 / 3) < 1e-12
assert abs((Hy - Hy_x) / Hx - 0.420620) < 1e-6

H(Y) = 1.000000   H(Y|X) = 0.333333   IG = 0.666667
H(X) = 1.584963   gain ratio = 0.420620


### Problem L2.3 — Fano floor for a ten-class classifier

**Statement.** A $10$-class problem has residual uncertainty $H(Y \mid X) = 1.5$ bits. Use
Theorem 4.8 in the relaxed form $H(Y \mid X) \le 1 + P_e \log(K-1)$ to lower-bound the error of
any classifier, then compare against the exact bound $H_b(P_e) + P_e\log(K-1)$.

**Intuition.** The binary entropy term is worth at most one bit, so everything above one bit of
residual entropy must be paid for in errors.

**Solution.**

*Step 1 — relaxed bound.* With $K = 10$ and $H_b(P_e) \le 1$,

$$
1.5 \le 1 + P_e \log 9 \implies P_e \ge \frac{0.5}{\log 9} = \frac{0.5}{3.169925} = 0.157732 .
$$

*Step 2 — exact bound.* Solving $H_b(P_e) + P_e \log 9 = 1.5$ numerically gives
$P_e \ge 0.228558$.

*Step 3 — read the gap.* The relaxed form understates the floor by $0.070825$, which is $31.0\%$
of the true floor: it throws away the true binary-entropy term
$H_b(0.228558) = 0.775489$ bits and replaces it by the crude bound $1$.

$$
\boxed{P_e \ge 0.157732 \text{ (relaxed)}, \qquad P_e \ge 0.228558 \text{ (exact Fano)}}
$$

**Key takeaway.** Fano turns residual conditional entropy into a hard floor on accuracy, and the
floor is a property of the joint distribution — no architecture, dataset size or compute budget
moves it. Quote the exact form: the relaxed one is visibly weaker.

In [14]:
from scipy.optimize import brentq

K, target = 10, 1.5
relaxed = (target - 1.0) / np.log2(K - 1)
exact = brentq(lambda t: Hb(t) + t * np.log2(K - 1) - target, 1e-12, 1.0 - 1e-12)
print(f"relaxed floor (H - 1)/log2(K-1) = {relaxed:.6f}")
print(f"exact Fano floor                = {exact:.6f}")
print(f"understatement                  = {exact - relaxed:.6f} "
      f"({100 * (exact - relaxed) / exact:.1f} percent of the true floor)")
print(f"Hb at the exact floor           = {Hb(exact):.6f} bits (the term the relaxation drops)")
assert abs(relaxed - 0.157732) < 1e-6 and abs(exact - 0.228558) < 1e-6
assert abs(Hb(exact) - 0.775489) < 1e-6

relaxed floor (H - 1)/log2(K-1) = 0.157732
exact Fano floor                = 0.228558
understatement                  = 0.070825 (31.0 percent of the true floor)
Hb at the exact floor           = 0.775489 bits (the term the relaxation drops)


### Problem L2.4 — Label noise raises the loss floor

**Statement.** Clean labels $Y \in \{0,1\}$ satisfy $H(Y \mid X) = 0$. They are corrupted into
$\tilde{Y}$ by a symmetric flip of rate $\epsilon = 0.1$, independent of $X$ given $Y$. Compute
$H(\tilde{Y} \mid X)$.

**Intuition.** With the clean label determined by $X$, the only thing left uncertain is the coin
that decides whether to flip it.

**Solution.**

*Step 1.* $H(Y \mid X) = 0$ gives $Y = f(X)$ almost surely by Theorem 4.7.

*Step 2.* Conditioned on $X = x$, the noisy label is $f(x)$ flipped with probability $\epsilon$,
so each slice is $\mathrm{Bernoulli}(\epsilon)$ up to relabelling and

$$
H(\tilde{Y} \mid X{=}x) = H_b(\epsilon) \text{ for every } x .
$$

*Step 3.* Averaging leaves $H(\tilde{Y} \mid X) = H_b(0.1) = 0.468996$ bits, i.e.
$0.325083$ nats.

$$
\boxed{H(\tilde{Y} \mid X) = H_b(0.1) = 0.468996 \text{ bits/example}}
$$

**Key takeaway.** The conditional entropy of the *label-generating process* is the honest floor of
the training loss; a run that drives loss below it is memorizing the noise realization, not
learning.

In [15]:
eps = 0.1
floor_bits = Hb(eps)
print(f"H(Ytilde | X) = Hb({eps}) = {floor_bits:.6f} bits = {floor_bits * np.log(2):.6f} nats")
# direct check on an explicit joint over (X, Ytilde) with f(x) = x mod 2
m = 6
PX = rng.dirichlet(np.ones(m))
P = np.zeros((m, 2))
for x in range(m):
    P[x, x % 2] = PX[x] * (1 - eps)
    P[x, 1 - x % 2] = PX[x] * eps
_, _, _, Hyt_x, _ = entropies(P)
print(f"explicit joint over {m} feature values: H(Ytilde|X) = {Hyt_x:.6f} bits")
assert abs(Hyt_x - floor_bits) < 1e-12
assert abs(floor_bits - 0.468996) < 1e-6

H(Ytilde | X) = Hb(0.1) = 0.468996 bits = 0.325083 nats
explicit joint over 6 feature values: H(Ytilde|X) = 0.468996 bits


### Problem L2.5 — Bits bought by longer context

**Statement.** A language model's measured conditional entropies are $H(X_t) = 4.1$,
$H(X_t \mid X_{t-1}) = 3.2$ and $H(X_t \mid X_{t-2}, X_{t-1}) = 2.8$ bits. How much does each
context extension save, and can longer context ever increase conditional entropy?

**Intuition.** Each extra conditioning variable can only shrink the entropy, and the shrinkage is
the successive difference.

**Solution.**

*Step 1.* Adding one token of context saves $4.1 - 3.2 = 0.9$ bits per token; adding the second
saves $3.2 - 2.8 = 0.4$ bits per token.

*Step 2.* No: Theorem 4.4(b) gives

$$
H(X_t \mid X_{t-2}, X_{t-1}) \le H(X_t \mid X_{t-1}) \le H(X_t)
$$

because each step conditions on a superset.

*Step 3 — the caveat.* The monotonicity is a statement about the *true* conditional entropies.
Plug-in estimates from finite data are biased downwards by roughly $(S-1)/(2N\ln 2)$ bits
(Section 7.2 of the theory notebook), and $S$ grows with the context window, so estimated
conditional entropies can and do violate the ordering.

$$
\boxed{0.9 \text{ then } 0.4 \text{ bits/token saved; the true } H \text{ is non-increasing in context}}
$$

**Key takeaway.** "More context never hurts" is exactly Theorem 4.4(b) — for the true
distribution. On finite samples it is an empirical question, and the bias runs the wrong way.

In [16]:
Hs = np.array([4.1, 3.2, 2.8])
print(f"savings per extension: {np.round(-np.diff(Hs), 4)} bits/token")
print(f"monotone non-increasing: {bool((np.diff(Hs) <= 0).all())}")
# the plug-in bias that can break the ordering empirically, for a V-symbol vocabulary
print("\nplug-in entropy bias -(S-1)/(2 N ln 2) for a table of S = V^(k+1) cells:")
for V, N in [(50, 10_000), (50, 100_000)]:
    for ctx in (0, 1, 2):
        S = V ** (ctx + 1)
        print(f"  V = {V}, N = {N}, context length {ctx}: S = {S:>7}, bias about "
              f"{-(S - 1) / (2 * N * np.log(2)):.4f} bits")
assert (np.diff(Hs) <= 0).all()

savings per extension: [0.9 0.4] bits/token
monotone non-increasing: True

plug-in entropy bias -(S-1)/(2 N ln 2) for a table of S = V^(k+1) cells:
  V = 50, N = 10000, context length 0: S =      50, bias about -0.0035 bits
  V = 50, N = 10000, context length 1: S =    2500, bias about -0.1803 bits
  V = 50, N = 10000, context length 2: S =  125000, bias about -9.0168 bits
  V = 50, N = 100000, context length 0: S =      50, bias about -0.0004 bits
  V = 50, N = 100000, context length 1: S =    2500, bias about -0.0180 bits
  V = 50, N = 100000, context length 2: S =  125000, bias about -0.9017 bits


### Problem L2.6 — Joint compression of correlated sensors

**Statement.** Two sensors have $H(X) = H(Y) = 6$ bits per reading and $H(X, Y) = 8$ bits.
Compare separate against joint lossless compression, and state precisely what Slepian-Wolf adds.

**Intuition.** The overlap is counted twice when the sensors are coded separately and once when
they are coded together.

**Solution.**

*Step 1.* Separate encoding and separate decoding needs $H(X) + H(Y) = 12$ bits per pair.

*Step 2.* Joint encoding needs $H(X, Y) = 8$ bits, a saving of $4$ bits per pair — exactly the
overlap $I(X;Y) = 12 - 8$ of Definition 3.5, and a $33.3\%$ reduction.

*Step 3.* The conditionals follow from Theorem 4.1: $H(Y \mid X) = 8 - 6 = 2$ bits and
$H(X \mid Y) = 2$ bits.

*Step 4 — Slepian-Wolf.* With **separate encoders** and a **joint decoder**, every rate pair with
$R_X \ge H(X\mid Y) = 2$, $R_Y \ge H(Y \mid X) = 2$ and $R_X + R_Y \ge H(X,Y) = 8$ is achievable
for long i.i.d. blocks. In particular $(6, 2)$ works: the second sensor may transmit at
$H(Y \mid X)$ without ever seeing $X$.

$$
\boxed{12 \text{ bits separately vs } H(X,Y) = 8 \text{ bits jointly; } (R_X, R_Y) = (6, 2) \text{ achievable with separate encoders}}
$$

**Key takeaway.** Correlation is compression, and the Slepian-Wolf theorem
(Cover & Thomas, 2nd ed., section 15.4, Theorem 15.4.1) says it survives the encoders being kept
apart — only the decoder must be joint.

In [17]:
Hx, Hy, Hxy = 6.0, 6.0, 8.0
I = Hx + Hy - Hxy
print(f"separate {Hx + Hy:.1f} bits   joint {Hxy:.1f} bits   overlap I(X;Y) = {I:.1f} bits")
print(f"saving   {100 * I / (Hx + Hy):.1f} percent")
print(f"H(Y|X) = {Hxy - Hx:.1f} bits   H(X|Y) = {Hxy - Hy:.1f} bits")
for Rx, Ry in [(6.0, 2.0), (2.0, 6.0), (4.0, 4.0), (2.0, 2.0), (5.0, 2.0)]:
    ok = (Rx >= Hxy - Hy) and (Ry >= Hxy - Hx) and (Rx + Ry >= Hxy)
    print(f"  rate pair ({Rx:.0f}, {Ry:.0f}) inside the Slepian-Wolf region: {ok}")
assert abs(I - 4.0) < 1e-12

separate 12.0 bits   joint 8.0 bits   overlap I(X;Y) = 4.0 bits
saving   33.3 percent
H(Y|X) = 2.0 bits   H(X|Y) = 2.0 bits
  rate pair (6, 2) inside the Slepian-Wolf region: True
  rate pair (2, 6) inside the Slepian-Wolf region: True
  rate pair (4, 4) inside the Slepian-Wolf region: True
  rate pair (2, 2) inside the Slepian-Wolf region: False
  rate pair (5, 2) inside the Slepian-Wolf region: False


### Problem L2.7 — Landauer's bill for erasing a gibibyte

**Statement.** A register holds $N = 2^{33}$ bits (one gibibyte). Compute the minimum heat
dissipated by resetting it to all zeros at $T = 300$ K when (a) the stored bits are i.i.d. fair,
and (b) they are i.i.d. $\mathrm{Bernoulli}(0.1)$. Use $k_B = 1.380649 \times 10^{-23}$ J/K and
Landauer's bound $W \ge k_B T \ln 2 \cdot H(M)$ with $H(M)$ in bits.

**Intuition.** Erasure destroys entropy, and Landauer prices entropy, not storage: a compressible
memory is cheaper to erase.

**Solution.**

*Step 1 — the constant.* $k_B T \ln 2 = (1.380649\times10^{-23})(300)(0.6931472) = 2.870979 \times 10^{-21}$ J
per bit of entropy.

*Step 2 — fair bits.* Independence and Theorem 4.5's equality case give
$H(M) = N \times 1 = 8{,}589{,}934{,}592$ bits, so

$$
W \ge (2.870979\times 10^{-21})(8.589935\times10^{9}) = 2.466152 \times 10^{-11} \text{ J}.
$$

*Step 3 — biased bits.* Now $H(M) = N \, H_b(0.1) = 2^{33}(0.468996) = 4.028641 \times 10^{9}$
bits, so

$$
W \ge (2.870979 \times 10^{-21})(4.028641 \times 10^{9}) = 1.156614 \times 10^{-11} \text{ J},
$$

a factor $H_b(0.1) = 0.468996$ of the fair-bit bill.

$$
\boxed{W_{\text{fair}} \ge 2.466152 \times 10^{-11} \text{ J}, \qquad W_{\mathrm{Ber}(0.1)} \ge 1.156614 \times 10^{-11} \text{ J}}
$$

**Key takeaway.** Landauer's cost scales with the **entropy** of the record, not with the number
of cells holding it — which is why a memory you could have compressed is a memory you could have
erased more cheaply.

In [18]:
N = 2 ** 33
per_bit = KB * TEMP * np.log(2.0)
print(f"kB T ln 2 = {per_bit:.6e} J per bit of entropy")
for name, h in [("i.i.d. fair    ", 1.0), ("i.i.d. Ber(0.1)", Hb(0.1))]:
    HM = N * h
    print(f"  {name}: H(M) = {HM:.6e} bits   W >= {per_bit * HM:.6e} J")
print(f"  ratio of the two bills = {Hb(0.1):.6f} = Hb(0.1)")
assert abs(per_bit * N * 1.0 - 2.466152e-11) < 1e-16
assert abs(per_bit * N * Hb(0.1) - 1.156614e-11) < 1e-16

kB T ln 2 = 2.870979e-21 J per bit of entropy
  i.i.d. fair    : H(M) = 8.589935e+09 bits   W >= 2.466152e-11 J
  i.i.d. Ber(0.1): H(M) = 4.028641e+09 bits   W >= 1.156614e-11 J
  ratio of the two bills = 0.468996 = Hb(0.1)


### Problem L2.8 — Maxwell's demon with a noisy thermometer

**Statement.** A one-bit system sits in state $X$, uniform on $\{0,1\}$. A demon measures it and
records $M$, which agrees with $X$ with probability $1 - \epsilon$. Using the feedback bound
$W_{\text{ext}} \le k_B T \, I(X;M)$ and Landauer's $W_{\text{erase}} \ge k_B T \, H(M)$, find the
maximum net work per cycle at $T = 300$ K for $\epsilon \in \{0, 0.05, 0.2\}$, and determine every
$\epsilon$ at which the demon breaks even.

**Intuition.** The demon can cash in only the *correlation* $I(X;M)$ but must erase the whole
record $H(M)$; the difference is its own measurement noise.

**Solution.**

*Step 1 — the record's marginal.* By symmetry $M$ is uniform, so $H(M) = 1$ bit.

*Step 2 — the noise.* $H(M \mid X) = H_b(\epsilon)$, since each slice is a $\mathrm{Bernoulli}(\epsilon)$
flip of the true state.

*Step 3 — the correlation.* By Theorem 4.1 and Definition 3.5,
$I(X; M) = H(M) - H(M \mid X) = 1 - H_b(\epsilon)$ bits.

*Step 4 — the budget.*

$$
W_{\text{net}} \le k_B T \bigl( I(X;M) - H(M) \bigr) = -\,k_B T \, H(M \mid X) = -\,k_B T \ln 2 \cdot H_b(\epsilon) \text{ joules}.
$$

*Step 5 — the numbers at $T = 300$ K.* With $k_B T \ln 2 = 2.870979 \times 10^{-21}$ J,

- $\epsilon = 0$: $W_{\text{net}} \le 0$ exactly;
- $\epsilon = 0.05$: $W_{\text{net}} \le -8.222 \times 10^{-22}$ J;
- $\epsilon = 0.2$: $W_{\text{net}} \le -2.073 \times 10^{-21}$ J.

*Step 6 — break-even.* $H_b(\epsilon) = 0$ only at $\epsilon \in \{0, 1\}$, and $\epsilon = 1$ is a
perfectly anti-correlated thermometer, which is noiseless after relabelling. So break-even happens
exactly when the measurement is deterministic, which by Theorem 4.7 is $H(M \mid X) = 0$.

$$
\boxed{W_{\text{net}} \le -\,k_B T \ln 2 \cdot H_b(\epsilon), \qquad \text{break-even} \iff \epsilon \in \{0, 1\}}
$$

**Key takeaway.** The demon's per-cycle deficit is exactly the entropy of its own measurement
noise. The second law is rescued not by forbidding measurement but by charging for erasure — and
the charge is a conditional entropy.

In [19]:
per_bit = KB * TEMP * np.log(2.0)
print(f"{'eps':>6} {'H(M|X)':>9} {'I(X;M)':>9} {'W_net max [J]':>16}")
for e in (0.0, 0.05, 0.2, 0.5, 1.0):
    HM_X = Hb(e)
    print(f"{e:>6.2f} {HM_X:>9.6f} {1 - HM_X:>9.6f} {-per_bit * HM_X:>16.4e}")
grid = np.linspace(0.0, 1.0, 1001)
breakeven = grid[np.array([Hb(t) for t in grid]) < 1e-12]
print(f"break-even eps values on a 1001-point grid: {breakeven}")
assert abs(-per_bit * Hb(0.05) + 8.2224e-22) < 1e-25
assert abs(-per_bit * Hb(0.2) + 2.0726e-21) < 1e-24
assert np.allclose(breakeven, [0.0, 1.0])

   eps    H(M|X)    I(X;M)    W_net max [J]
  0.00  0.000000  1.000000      -0.0000e+00
  0.05  0.286397  0.713603      -8.2224e-22
  0.20  0.721928  0.278072      -2.0726e-21
  0.50  1.000000  0.000000      -2.8710e-21
  1.00  0.000000  1.000000      -0.0000e+00
break-even eps values on a 1001-point grid: [0. 1.]


## L3 — Challenge Proofs

### Problem L3.1 — Han's inequality for three variables, and its equality case

**Statement.** For any $X_1, X_2, X_3$ on finite alphabets prove

$$
H(X_1, X_2, X_3) \le \tfrac12 \bigl[ H(X_1, X_2) + H(X_2, X_3) + H(X_1, X_3) \bigr],
$$

and show that equality holds **if and only if** $X_1, X_2, X_3$ are mutually independent.
(Theorem 4.9 proves the inequality for general $n$ and the "if" direction; the "only if" is the
new content here.)

**Intuition.** Each pair sees two of the three variables, so summing the three pairs counts every
variable twice — but conditioning on more can only shrink, and that shrinkage is what pays for
the second copy.

**Solution.**

*Step 1 — write each pair as the triple minus one conditional.* Apply Theorem 4.1 to the pair
consisting of the two retained variables and the missing one:

$$
H(X_1, X_2) = H(X_1, X_2, X_3) - H(X_3 \mid X_1, X_2),
$$

$$
H(X_1, X_3) = H(X_1, X_2, X_3) - H(X_2 \mid X_1, X_3),
$$

$$
H(X_2, X_3) = H(X_1, X_2, X_3) - H(X_1 \mid X_2, X_3).
$$

*Step 2 — two genuine inequalities.* Theorem 4.4(b) enlarges each conditioning set to the one used
by the chain rule in the order $1, 2, 3$:

$$
H(X_1 \mid X_2, X_3) \le H(X_1),
$$

$$
H(X_2 \mid X_1, X_3) \le H(X_2 \mid X_1) .
$$

The third term, $H(X_3 \mid X_1, X_2)$, is already conditioned on everything preceding it and
needs no bound.

*Step 3 — sum.* Writing $T = H(X_1,X_2,X_3)$ and adding the three lines of Step 1,

$$
\sum_{\text{pairs}} H = 3T - \bigl[ H(X_1 \mid X_2,X_3) + H(X_2 \mid X_1, X_3) + H(X_3 \mid X_1,X_2) \bigr] .
$$

By Step 2 the bracket is at most $H(X_1) + H(X_2 \mid X_1) + H(X_3 \mid X_1, X_2) = T$, using
Theorem 4.2. Hence $\sum_{\text{pairs}} H \ge 3T - T = 2T$, which is the claim.

*Step 4 — equality forces $X_1$ independent of the rest.* Equality needs both inequalities of
Step 2 to be equalities. The first is $H(X_1 \mid X_2, X_3) = H(X_1)$, which by Theorem 4.4(a)
applied with the single variable $(X_2, X_3)$ says

$$
p(x_1, x_2, x_3) = p(x_1)\, p(x_2, x_3) \quad \text{for all } (x_1,x_2,x_3).
$$

*Step 5 — and then $X_2$ independent of $X_3$.* The second equality is
$H(X_2 \mid X_1, X_3) = H(X_2 \mid X_1)$. By Step 4, conditioned on any $x_1$ with
$p(x_1) \gt 0$ the pair $(X_2, X_3)$ has law $p(x_2, x_3)$, not depending on $x_1$; so both sides
reduce to the unconditional quantities and the equality reads $H(X_2 \mid X_3) = H(X_2)$, i.e.
$X_2 \perp X_3$ by Theorem 4.4(a).

*Step 6 — combine.* Steps 4 and 5 give
$p(x_1,x_2,x_3) = p(x_1)p(x_2,x_3) = p(x_1)p(x_2)p(x_3)$: mutual independence. The converse is
Theorem 4.9's equality clause.

$$
\boxed{H(X_1,X_2,X_3) \le \tfrac12 \sum_{\text{pairs}} H(\text{pair}), \quad \text{equality} \iff X_1, X_2, X_3 \text{ mutually independent}}
$$

**Key takeaway.** Han's inequality is subadditivity applied one variable at a time, and — exactly
like subadditivity — it is tight only at full mutual independence. The XOR triple is pairwise
independent, has all three pairs at the independent value, and still misses by a full bit.

In [20]:
def han_gap(J):
    """LHS minus RHS of Han's inequality for a 3-way joint array."""
    pairs = [H(J.sum(2)), H(J.sum(1)), H(J.sum(0))]
    return H(J) - sum(pairs) / 2.0


Jxor = np.zeros((2, 2, 2))
for a in (0, 1):
    for b in (0, 1):
        Jxor[a, b, a ^ b] = 0.25
Jiid = np.full((2, 2, 2), 0.125)
Jdep = np.zeros((2, 2, 2))
for a in (0, 1):
    Jdep[a, a, a] = 0.5                       # all three equal

print(f"{'case':<22} {'H(triple)':>10} {'Han RHS':>10} {'gap':>10}")
for name, J in [("XOR (pairwise indep)", Jxor), ("i.i.d. fair bits", Jiid),
                ("all three equal", Jdep)]:
    pairs = [H(J.sum(2)), H(J.sum(1)), H(J.sum(0))]
    print(f"{name:<22} {H(J):>10.6f} {sum(pairs) / 2:>10.6f} {han_gap(J):>10.6f}")
worst = 0.0
for _ in range(300):
    J = rng.random((3, 2, 4))
    J /= J.sum()
    worst = max(worst, han_gap(J))
    assert han_gap(J) <= 1e-12
print(f"\n300 random 3x2x4 joints: worst LHS - RHS = {worst:.3e} (must be <= 0)")
assert abs(han_gap(Jiid)) < 1e-12 and han_gap(Jxor) < -0.9

case                    H(triple)    Han RHS        gap
XOR (pairwise indep)     2.000000   3.000000  -1.000000
i.i.d. fair bits         3.000000   3.000000   0.000000
all three equal          1.000000   1.500000  -0.500000

300 random 3x2x4 joints: worst LHS - RHS = 0.000e+00 (must be <= 0)


### Problem L3.2 — The entropy rate of a stationary process exists

**Statement.** For a stationary process $(X_t)$ on a finite alphabet, show that
$a_n = H(X_n \mid X_1, \dots, X_{n-1})$ is non-increasing, and conclude via Cesàro means that
$\lim_n \frac1n H(X_1, \dots, X_n)$ exists and equals $\lim_n a_n$.

**Intuition.** More history can only help, and stationarity says the history looks the same
wherever you stand — so the per-step cost can only fall, and the running average follows it down.

**Solution.**

*Step 1 — monotonicity.* Theorem 4.4(b) drops $X_1$ from the conditioning set, then stationarity
shifts every index down by one:

$$
a_{n+1} = H(X_{n+1} \mid X_1, \dots, X_n) \le H(X_{n+1} \mid X_2, \dots, X_n) = H(X_n \mid X_1, \dots, X_{n-1}) = a_n .
$$

*Step 2 — convergence of $a_n$.* The sequence is non-increasing and bounded below by $0$
(Proposition 4.6), hence converges to some $H_\infty \ge 0$.

*Step 3 — the running average.* Theorem 4.2 gives
$\frac1n H(X_{1:n}) = \frac1n \sum_{k=1}^{n} a_k$.

*Step 4 — Cesàro.* Fix $\varepsilon \gt 0$ and choose $N$ with
$\lvert a_k - H_\infty \rvert \lt \varepsilon$ for $k \gt N$. Then for $n \gt N$,

$$
\left\lvert \frac1n \sum_{k=1}^{n} a_k - H_\infty \right\rvert \le \frac{C_N}{n} + \varepsilon,
\qquad C_N = \sum_{k \le N} \lvert a_k - H_\infty \rvert ,
$$

so the $\limsup$ is at most $\varepsilon$; as $\varepsilon$ was arbitrary the limit is
$H_\infty$.

$$
\boxed{H_\infty = \lim_n H(X_n \mid X_{\lt n}) = \lim_n \tfrac1n H(X_{1:n}) \text{ exists for every stationary process}}
$$

**Key takeaway.** "Bits per symbol" is a genuine asymptotic constant for stationary sources — the
number every compressor and every language-model scaling curve is chasing. Problem L1.6 shows both
halves fail without stationarity.

In [21]:
Pm = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.25, 0.25, 0.5]])
w, V = np.linalg.eig(Pm.T)
pi = np.real(V[:, int(np.argmin(np.abs(w - 1.0)))])
pi = pi / pi.sum()
rate = float(pi @ np.array([H(Pm[i]) for i in range(3)]))

# exact trajectory distribution over 3**n states, grown one step at a time
traj = pi.copy()
Hs = [H(traj)]
for n in range(2, 14):
    traj = (traj.reshape(-1, 3)[:, :, None] * Pm[None, :, :]).reshape(-1)
    Hs.append(H(traj))
Hs = np.array(Hs)
a_seq = np.concatenate([[Hs[0]], np.diff(Hs)])
avg_seq = Hs / np.arange(1, len(Hs) + 1)

print(f"stationary pi = {pi}   (pi P = pi: {np.allclose(pi @ Pm, pi)})")
print(f"predicted entropy rate = {rate:.6f} bits/step\n")
print(f"{'n':>4} {'a_n':>10} {'(1/n) H':>10}")
for n in range(1, len(Hs) + 1):
    print(f"{n:>4} {a_seq[n - 1]:>10.6f} {avg_seq[n - 1]:>10.6f}")
assert np.allclose(pi @ Pm, pi)
assert all(a_seq[i + 1] <= a_seq[i] + 1e-12 for i in range(len(a_seq) - 1))
assert abs(a_seq[-1] - rate) < 1e-9
assert abs(avg_seq[-1] - rate) < 0.05

stationary pi = [0.3125 0.5208 0.1667]   (pi P = pi: True)
predicted entropy rate = 1.091665 bits/step

   n        a_n    (1/n) H
   1   1.445384   1.445384
   2   1.091665   1.268524
   3   1.091665   1.209571
   4   1.091665   1.180094
   5   1.091665   1.162408
   6   1.091665   1.150618
   7   1.091665   1.142196
   8   1.091665   1.135879
   9   1.091665   1.130967
  10   1.091665   1.127036
  11   1.091665   1.123821
  12   1.091665   1.121141
  13   1.091665   1.118874


### Problem L3.3 — Conditional subadditivity and a Venn caution

**Statement.** (a) Prove $H(X, Y \mid Z) \le H(X \mid Z) + H(Y \mid Z)$. (b) Exhibit three
variables for which the triple overlap $I(X;Y) - I(X;Y \mid Z)$ is negative, so entropy diagrams
can have negative central regions.

**Intuition.** Subadditivity holds inside every slice, and averaging preserves it; but the
*interaction* of three variables is a difference of two non-negative numbers and has no sign.

**Solution.**

*Step 1 — part (a), fix $z$.* For each $z$ with $p(z) \gt 0$, apply Theorem 4.5 to the conditional
joint law $p(x, y \mid z)$:

$$
H(X, Y \mid Z{=}z) \le H(X \mid Z{=}z) + H(Y \mid Z{=}z) .
$$

*Step 2 — average.* Multiply by $p(z)$ and sum; each term becomes the corresponding conditional
entropy of Definition 3.3, giving the claim. Equality holds iff $X \perp Y$ given $Z{=}z$ for
every such $z$.

*Step 3 — part (b), the XOR triple.* Let $X, Y$ be independent fair bits and $Z = X \oplus Y$.
Independence gives $I(X;Y) = 0$.

*Step 4 — condition on $Z$.* Given $Z$, knowing $X$ determines $Y = X \oplus Z$, so
$H(Y \mid X, Z) = 0$ while $H(Y \mid Z) = 1$ bit (given $Z$, $Y$ is still a fair bit). Hence

$$
I(X;Y \mid Z) = H(Y \mid Z) - H(Y \mid X, Z) = 1 - 0 = 1 \text{ bit}.
$$

*Step 5 — the interaction.* $I(X;Y) - I(X;Y \mid Z) = 0 - 1 = -1$ bit.

$$
\boxed{H(X,Y \mid Z) \le H(X \mid Z) + H(Y \mid Z); \qquad \text{XOR gives interaction } -1 \text{ bit}}
$$

**Key takeaway.** Conditioning can *create* dependence — the "explaining away" of Bayesian
networks. Two-variable entropy diagrams are exact; the three-circle picture is a mnemonic whose
central region can be negative and therefore is not an area.

In [22]:
Jxor = np.zeros((2, 2, 2))
for a in (0, 1):
    for b in (0, 1):
        Jxor[a, b, a ^ b] = 0.25
Hz = H(Jxor.sum((0, 1)))
Hxy_z = H(Jxor) - Hz
Hx_z = H(Jxor.sum(1)) - Hz
Hy_z = H(Jxor.sum(0)) - Hz
I_xy = H(Jxor.sum((1, 2))) + H(Jxor.sum((0, 2))) - H(Jxor.sum(2))
I_xy_z = Hx_z + Hy_z - Hxy_z
print(f"(a) H(X,Y|Z) = {Hxy_z:.6f} <= H(X|Z) + H(Y|Z) = {Hx_z + Hy_z:.6f}")
print(f"(b) I(X;Y) = {I_xy:.6f}   I(X;Y|Z) = {I_xy_z:.6f}   interaction = {I_xy - I_xy_z:.6f}")
worst = 0.0
for _ in range(300):
    J = rng.random((3, 3, 2))
    J /= J.sum()
    Hz_ = H(J.sum((0, 1)))
    worst = max(worst, (H(J) - Hz_) - (H(J.sum(1)) - Hz_) - (H(J.sum(0)) - Hz_))
print(f"300 random joints: worst H(X,Y|Z) - H(X|Z) - H(Y|Z) = {worst:.3e} (must be <= 0)")
assert Hxy_z <= Hx_z + Hy_z + 1e-12
assert abs(I_xy) < 1e-12 and abs(I_xy_z - 1.0) < 1e-12 and worst <= 1e-12

(a) H(X,Y|Z) = 1.000000 <= H(X|Z) + H(Y|Z) = 2.000000
(b) I(X;Y) = 0.000000   I(X;Y|Z) = 1.000000   interaction = -1.000000
300 random joints: worst H(X,Y|Z) - H(X|Z) - H(Y|Z) = 0.000e+00 (must be <= 0)


### Problem L3.4 — Shearer's lemma and counting triangles

**Statement.** (a) Prove **Shearer's lemma**: if $S_1, \dots, S_m \subseteq \{1, \dots, n\}$ are
such that every index $i$ lies in at least $k$ of them, then

$$
k \, H(X_1, \dots, X_n) \le \sum_{j=1}^{m} H\bigl(X_{S_j}\bigr).
$$

(b) Use it to prove that a graph with $e \ge 1$ edges and $t \ge 1$ triangles satisfies
$t \le (2e)^{3/2}/6$.

**Intuition.** Shearer says a joint distribution cannot be much richer than what its projections
see. Applied to a random triangle, the three vertex pairs are edges, so there cannot be many more
triangles than the edge count allows.

**Solution.**

*Step 0 — a sub-tuple bound.* Let $S = \{i_1 \lt \dots \lt i_r\}$. The chain rule of Theorem 4.2
applied inside $S$, followed by Theorem 4.4(b) enlarging each conditioning set from
$\{i_1,\dots,i_{l-1}\}$ to $\{1, \dots, i_l - 1\}$, gives

$$
H(X_S) = \sum_{l=1}^{r} H\bigl(X_{i_l} \mid X_{i_1}, \dots, X_{i_{l-1}}\bigr)
\ \ge \ \sum_{i \in S} H\bigl(X_i \mid X_{\lt i}\bigr) .
$$

*Step 1 — sum over the subsets.* Adding that bound over $j = 1, \dots, m$ and exchanging the
order of summation,

$$
\sum_{j=1}^{m} H(X_{S_j}) \ \ge \ \sum_{i=1}^{n} \bigl\lvert \{ j : i \in S_j \} \bigr\rvert \, H(X_i \mid X_{\lt i})
\ \ge \ k \sum_{i=1}^{n} H(X_i \mid X_{\lt i}) = k\, H(X_1, \dots, X_n),
$$

the last equality by Theorem 4.2. That is Shearer's lemma.

*Step 2 — the random triangle.* Assume $t \ge 1$ (for $t = 0$ the bound is trivial) and pick one
of the $6t$ **ordered** triangles $(V_1, V_2, V_3)$ uniformly at random. A uniform variable on a
support of size $6t$ has entropy exactly $\log(6t)$ by the maximum-entropy corollary of Lemma 4.3,
so

$$
H(V_1, V_2, V_3) = \log (6t) .
$$

*Step 3 — apply Shearer with the three pairs.* Take $S_1 = \{1,2\}$, $S_2 = \{2,3\}$,
$S_3 = \{1,3\}$; each index lies in exactly $k = 2$ of them, so

$$
2 \log (6t) \le H(V_1,V_2) + H(V_2,V_3) + H(V_1,V_3) .
$$

*Step 4 — bound each pair by the edge count.* Each pair $(V_i, V_j)$ is supported on the ordered
edges of the graph, of which there are $2e$, so again by the maximum-entropy corollary
$H(V_i, V_j) \le \log(2e)$. Hence

$$
2 \log (6t) \le 3 \log (2e) \implies (6t)^2 \le (2e)^3 \implies t \le \frac{(2e)^{3/2}}{6} .
$$

$$
\boxed{k\,H(X_{[n]}) \le \sum_j H(X_{S_j}); \qquad t \le \frac{(2e)^{3/2}}{6} = O\bigl(e^{3/2}\bigr)}
$$

**Key takeaway.** An entropy inequality proves a purely combinatorial theorem. Shearer converts
"every projection is small" into "the object is small" — the Loomis-Whitney and Kruskal-Katona
phenomenon in information-theoretic clothing — and it rests on nothing beyond Theorems 4.2 and
4.4(b) of this module.

In [23]:
def triangle_check(A):
    """Edges, triangles and the Shearer bound for a 0/1 symmetric adjacency matrix."""
    e = int(A.sum() // 2)
    t = int(round(np.trace(np.linalg.matrix_power(A, 3)) / 6))
    bound = (2 * e) ** 1.5 / 6 if e > 0 else 0.0
    return e, t, bound


print(f"{'graph':<26} {'e':>5} {'t':>7} {'(2e)^1.5/6':>12} {'holds':>6}")
graphs = {}
for nv in (4, 6, 10, 20):
    A = np.ones((nv, nv)) - np.eye(nv)
    graphs[f"complete K{nv}"] = A
for nv, pedge in [(30, 0.2), (30, 0.5), (60, 0.1)]:
    U = (rng.random((nv, nv)) < pedge).astype(float)
    A = np.triu(U, 1)
    graphs[f"random G({nv}, {pedge})"] = A + A.T
cyc = np.zeros((8, 8))
for i in range(8):
    cyc[i, (i + 1) % 8] = cyc[(i + 1) % 8, i] = 1.0
graphs["cycle C8 (t = 0)"] = cyc
for name, A in graphs.items():
    e, t, bound = triangle_check(A)
    print(f"{name:<26} {e:>5} {t:>7} {bound:>12.2f} {str(t <= bound + 1e-9):>6}")
    assert t <= bound + 1e-9
print("\nShearer's lemma on random joints (subsets {1,2}, {2,3}, {1,3}, k = 2):")
worst = 0.0
for _ in range(300):
    J = rng.random((3, 3, 3))
    J /= J.sum()
    lhs = 2 * H(J)
    rhs = H(J.sum(2)) + H(J.sum(0)) + H(J.sum(1))
    worst = max(worst, lhs - rhs)
    assert lhs <= rhs + 1e-12
print(f"  worst k*H(X) - sum H(X_S) = {worst:.3e} (must be <= 0)")

graph                          e       t   (2e)^1.5/6  holds
complete K4                    6       4         6.93   True
complete K6                   15      20        27.39   True
complete K10                  45     120       142.30   True
complete K20                 190    1140      1234.59   True
random G(30, 0.2)             86      31       375.96   True
random G(30, 0.5)            228     605      1622.92   True
random G(60, 0.1)            204      63      1373.53   True
cycle C8 (t = 0)               8       0        10.67   True

Shearer's lemma on random joints (subsets {1,2}, {2,3}, {1,3}, k = 2):
  worst k*H(X) - sum H(X_S) = 0.000e+00 (must be <= 0)
